# Session 5 · **The DilSeVector Concierge** ☕💘
### Prasad's app can find people. Today it learns to talk — and to stop making things up.

**The story:** DilSeVector now has a chat concierge that answers coffee questions from our real pages. It starts as an intern that invents cafés that don't exist. By the end it will remember you, send your question to the right desk, and **check its own work before it speaks.**

| Level | The concierge… | Called |
|---|---|---|
| L0 | …makes things up | (no RAG) |
| L1 | …reads the manual first | **Basic RAG** |
| L2 | …follows a written script | **Prompt template** |
| L3 | …remembers you | **Memory RAG** |
| L4 | …has specialist desks | **Branched RAG** |
| L5 | …spends only when needed | **Adaptive RAG** |
| L6 | …checks its own work | **Corrective RAG** |

---
### 📘 A note on the code in this notebook
Every function here uses only things you already know: **`for` loops, `if` statements, lists, dictionaries, and f-strings.** No lambdas, no comprehensions, no classes, no regular expressions.

That's deliberate. RAG is not hard — it's five steps in a row. If the Python gets clever, the idea gets hidden. Read every function top to bottom; there is nothing tricky in any of them.

---
### 🖥️ Companion local version
A local-VS-Code version of this same notebook (a `.py` script + `requirements.txt` + `README.md`) is available alongside this notebook if you'd rather run it outside Colab.


## Cell 1 — Setup
Run this once per Colab session. This connects to:
- **Gemini** — the language model that writes the concierge's answers
- **Pinecone** — the vector database that stores and searches our coffee pages
- **Sentence-Transformers** — the local embedding model that turns text into vectors

**Before running:** add your two secrets in Colab's 🔑 **Secrets** panel (left sidebar):
- `GOOGLE_API_KEY` — from https://aistudio.google.com/app/apikey
- `PINECONE_API_KEY` — from https://app.pinecone.io/

You'll also need the `coffee_pages/` folder of source HTML files sitting next to this notebook (download link in the next cell).

In [ ]:
## Download the coffee_pages source files (the HTML pages the concierge searches over).
## Download from Google Drive and place the "coffee_pages" folder in this Colab session's
## working directory (e.g. via the Files panel on the left, or by mounting Google Drive):
# https://drive.google.com/drive/folders/1wlvoSwA5H_JtU_lb0rSzDakPxZanhkmU?usp=sharing


In [ ]:
# Install the packages this notebook needs (quiet mode: -q hides the pip logs).
!pip -q install pinecone sentence-transformers google-generativeai beautifulsoup4

import os
import glob
import time
from bs4 import BeautifulSoup                        # parses the coffee-page HTML files
from sentence_transformers import SentenceTransformer  # turns text into embedding vectors
from pinecone import Pinecone                          # client for the Pinecone vector database
import google.generativeai as genai                    # client for the Gemini language model
from google.colab import userdata                       # reads secrets from Colab's Secrets panel

# --- Load secrets from Colab's Secrets panel (left sidebar -> key icon) ---
gemini_key = userdata.get('GOOGLE_API_KEY')
if not gemini_key:
    raise RuntimeError(
        "GOOGLE_API_KEY not found. "
        "Add it via Colab → 🔑 Secrets → enable Notebook access."
    )

pinecone_key = userdata.get('PINECONE_API_KEY')
if not pinecone_key:
    raise RuntimeError(
        "PINECONE_API_KEY not found. "
        "Add it via Colab → 🔑 Secrets → enable Notebook access."
    )

# --- Connect to Gemini (the language model that writes our answers) ---
genai.configure(api_key=gemini_key)
model = genai.GenerativeModel("gemini-3.6-flash")

# --- Load the embedding model (turns text into a 384-dim vector) ---
# This must match whatever dimension your Pinecone index ("coffeeindex") was created with.
embedder = SentenceTransformer("paraphrase-MiniLM-L6-v2")

# --- Connect to Pinecone and get a handle to the existing index ---
pc = Pinecone(api_key=pinecone_key)
index = pc.Index("coffeeindex")

# Pinecone "namespace" - a logical partition inside the index, like a table/collection.
NAMESPACE = "docs"

print("Ready. Embedding size:", embedder.get_sentence_embedding_dimension())


## Cell 2 — Read the coffee pages
Each of our pages has the same 8 sections (Overview, Ingredients, Method, and so on). So we make **one chunk per section**. That gives small, tidy pieces instead of one huge blob per page.

In [ ]:
def read_one_page(filename):
    """Turn one HTML file into a list of section pieces (chunks)."""
    html = open(filename, encoding="utf-8", errors="ignore").read()
    soup = BeautifulSoup(html, "html.parser")

    # Strip out anything that isn't real page content before extracting text.
    for tag in soup(["script", "style", "nav", "footer", "head"]):
        tag.decompose()

    title = soup.find("h1").get_text(strip=True)
    page_id = os.path.basename(filename).replace(".html", "")

    pieces = []
    for heading in soup.find_all("h2"):   # each <h2> starts a new section
        section_name = heading.get_text(strip=True)
        section_text = ""
        # Walk forward through the HTML from this heading until the NEXT <h2>,
        # collecting all the text in between as this section's content.
        for item in heading.next_siblings:
            if getattr(item, "name", None) == "h2":
                break
            if hasattr(item, "get_text"):
                section_text = section_text + " " + item.get_text(" ", strip=True)
        section_text = " ".join(section_text.split())   # collapse extra whitespace

        # Skip tiny/empty sections - not worth storing as a searchable piece.
        if len(section_text) > 40:
            piece = {}
            piece["id"] = page_id + "__" + section_name[:12]
            piece["page"] = page_id
            piece["title"] = title
            piece["section"] = section_name
            piece["text"] = title + " - " + section_name + ". " + section_text
            pieces.append(piece)
    return pieces


all_pieces = []
for filename in sorted(glob.glob("coffee_pages/*.html")):
    for piece in read_one_page(filename):
        all_pieces.append(piece)

print("Total pieces:", len(all_pieces))
print("Example:", all_pieces[0]["id"], "|", all_pieces[0]["section"])


## Cell 3 — Put them in Pinecone
Run once. Set `UPLOAD = True` the first time, then back to `False` (the data stays in Pinecone between sessions, so re-uploading every run just wastes API calls).

In [ ]:
UPLOAD = False   # set True only the FIRST time you run this notebook

if UPLOAD:
    rows = []
    for piece in all_pieces:
        vector = embedder.encode(piece["text"]).tolist()
        info = {"page": piece["page"], "title": piece["title"],
                "section": piece["section"], "text": piece["text"]}
        rows.append((piece["id"], vector, info))   # (id, vector, metadata) tuple per piece

    # Upload in batches of 50 (Pinecone limits how many vectors you can send per call).
    start = 0
    while start < len(rows):
        index.upsert(vectors=rows[start:start + 50], namespace="docs")
        start = start + 50
    time.sleep(5)   # give Pinecone a moment to finish indexing
    print("Uploaded", len(rows), "pieces")
else:
    print("Skipping upload")

print(index.describe_index_stats())   # confirms how many vectors are stored


## Cell 4 — 🎬 LEVEL 0: the intern who makes things up
No retrieval. Just ask the model.

🎲 **Guess first:** will it say "I don't know", or will it invent café names?

In [ ]:
# No RAG at all - just send the raw question straight to the language model.
# This is the baseline failure mode: with no real source of truth to draw from,
# the model will often confidently invent cafes, addresses, and facts that don't exist.
question = "Which cafes in Mumbai serve ashwagandha coffee? Give me names and addresses."

reply = model.generate_content(question)
print(reply.text)


## Cell 5 — 🪜 LEVEL 1: Basic RAG
Four steps. That is all RAG is.

1. turn the question into a vector
2. find the closest pieces
3. paste those pieces into a prompt
4. ask the model

In [ ]:
def search(question, how_many=4):
    """Step 1 and 2: turn the question into a vector, then find the closest pieces."""
    question_vector = embedder.encode(question).tolist()
    result = index.query(vector=question_vector, top_k=how_many,
                         namespace="docs", include_metadata=True)

    hits = []
    for match in result["matches"]:
        hit = {}
        hit["id"] = match["id"]
        hit["score"] = round(match["score"], 3)
        hit["title"] = match["metadata"]["title"]
        hit["section"] = match["metadata"]["section"]
        hit["text"] = match["metadata"]["text"]
        hits.append(hit)
    return hits


def format_context(hits):
    """Step 3: turn the retrieved pieces into text we can paste into a prompt."""
    lines = []
    for hit in hits:
        lines.append("[" + hit["id"] + "] " + hit["text"])
    return "\n\n".join(lines)


def build_prompt1(question, hits):
    """Step 4 (v1): build the prompt using an f-string, combining the question and context."""
    prompt = f"""Answer the question using ONLY the context below.
If the context does not contain the answer, say you do not know.

CONTEXT:
{format_context(hits)}

QUESTION: {question}
ANSWER:"""
    return prompt


def ask_model(prompt):
    """Step 4: send the finished prompt to the language model and return its reply text."""
    reply = model.generate_content(prompt)
    return reply.text


question = "What are the ingredients of ashwagandha coffee?"
#question = "How to make Tulsi coffee! "
hits = search(question)
print("Found these pieces:")
for hit in hits:
    print("  ", hit["score"], hit["id"])

# Build the prompt using the new function
prompt = build_prompt1(question, hits)

print("\nANSWER:")
print(ask_model(prompt))


## Cell 6 — 🪜 LEVEL 2: write the prompt down once
Above, the prompt was buried in the middle of our code. Now we make it a **template** we fill in — one place to edit, easy to improve.

The most important line is the last rule: **"if the context does not answer it, say so."** Without that line the model quietly makes things up.

In [ ]:
TEMPLATE = """You are the DilSeVector Concierge, a friendly coffee assistant.

RULES:
- Answer ONLY from the CONTEXT below.
- Put the [id] at the end of every sentence that states a fact.
- If the CONTEXT does not answer the question, say you do not know. Never guess.
- Keep it under 80 words.

THINGS WE KNOW ABOUT THIS USER:
{memory}

CONVERSATION SO FAR:
{history}

CONTEXT:
{context}

QUESTION: {question}
ANSWER:"""


def build_prompt(question, hits, memory_text="none", history_text="none"):
    """Fill in the shared TEMPLATE with this turn's question, context, memory, and history."""
    return TEMPLATE.format(memory=memory_text, history=history_text,
                           context=format_context(hits), question=question)


def answer(question, how_many=4):
    """Convenience wrapper: search, build the templated prompt, and ask the model."""
    hits = search(question, how_many)
    prompt = build_prompt(question, hits)
    return ask_model(prompt)


print(answer("Is turmeric coffee caffeine free?"))


## Cell 7 — 🪜 LEVEL 3: memory, and the word "it"
Ask two questions in a row:

1. *"Tell me about ashwagandha coffee."*
2. *"Compare **it** with turmeric coffee."*

🎲 **Guess first:** on question 2, what text does the search actually look for? The word "it" means nothing to the search — ashwagandha is nowhere in that sentence.

In [ ]:
print("Searching for just: 'Compare it with turmeric coffee'")
for hit in search("Compare it with turmeric coffee", 3):
    print("  ", hit["page"] if "page" in hit else hit["id"])

print("\nSearching with the earlier question glued on:")
better = "Tell me about ashwagandha coffee. Compare it with turmeric coffee"
for hit in search(better, 3):
    print("  ", hit["id"])


## Cell 8 — Two kinds of memory
- **History** = the chat so far. We just keep the last few turns in a list.
- **Preferences** = facts about the person ("I prefer low caffeine"). We keep those in a separate list.

Both get pasted into the prompt, in their own labelled slots — never mixed into the context.

In [ ]:
history = []       # list of ["User", "..."] / ["Assistant", "..."] pairs
preferences = []   # list of sentences describing user preferences


def find_preferences(text):
    """Pick out sentences where the user tells us what they like."""
    facts = []
    sentences = text.replace("!", ".").replace("?", ".").split(".")
    for sentence in sentences:
        clean = sentence.strip()
        lower = clean.lower()
        if lower.startswith("i prefer") or lower.startswith("i avoid") or lower.startswith("i like"):
            print("Fact:", clean)
            facts.append(clean)
    return facts


def format_history():
    """Turn the last few turns of `history` into text for the prompt's {history} slot."""
    if len(history) == 0:
        return "none"
    lines = []
    # Grab the last 4 items of the history
    for turn in history[-4:]:
        lines.append(turn[0] + ": " + turn[1])
    return "\n".join(lines)


def format_preferences():
    """Turn the stored `preferences` list into text for the prompt's {memory} slot."""
    if len(preferences) == 0:
        return "none"
    lines = []
    for fact in preferences:
        lines.append("- " + fact)
    return "\n".join(lines)


print(find_preferences("I prefer low caffeine drinks. I avoid bitter coffee."))
print(find_preferences("What is turmeric coffee?"))


## Cell 9 — One full turn of conversation
Same five steps every time: save preferences → search → build prompt → ask → remember.

Notice we **search again on every turn.** Memory keeps the conversation going; searching keeps the answer true.

In [ ]:
# This is the full conversational orchestration function that ties together everything
# you've built so far - preference tracking, retrieval, prompting, and history - into a
# multi-turn chat loop.
def chat(user_text):
    # 1. remember any preference the user just told us
    for fact in find_preferences(user_text):
        if fact not in preferences:
            preferences.append(fact)

    # 2. search (glue on the last question so words like "it" still work)
    search_text = user_text
    if len(history) > 0:
        search_text = history[-2][1] + " " + user_text if len(history) >= 2 else user_text

    hits = search(search_text, 4)

    # 3 and 4. build the prompt and ask
    format_preferences1 = format_preferences()
    print("format_preferences: ", format_preferences1)

    format_history1 = format_history()
    print("format_history: ", format_history1)

    prompt = build_prompt(user_text, hits, format_preferences1, format_history1)
    reply = ask_model(prompt)

    # 5. remember this turn
    history.append(["User", user_text])
    history.append(["Assistant", reply])

    print("  (used pieces:", end=" ")
    for hit in hits:
        print(hit["id"][:28], end=" ")
    print(")")
    return reply


questions = ["Tell me about ashwagandha coffee.",
             "I prefer low caffeine drinks.",
             "Compare it with turmeric coffee."]

for q in questions:
    print("\n👤 Question:", q)
    print("🤖 Answer: ", chat(q))


## Cell 10 — 🪜 LEVEL 4: specialist desks (Branched RAG)
Different questions need different treatment:

- a **comparison** must find pieces about **both** drinks, so ask for more pieces
- a **recommendation** needs the user's preferences
- a **lookup** wants one exact number, so fewer pieces is better

So: look at the question first, then decide how to search.

In [ ]:
def route(question):
    """Classify a question into one of: compare / recommend / lookup / explain."""
    q = question.lower()
    if "compare" in q or " vs " in q or "difference" in q:
        return "compare"
    if "for me" in q or "suggest" in q or "recommend" in q:
        return "recommend"
    if "how much" in q or "ratio" in q or "how long" in q or "how many" in q:
        return "lookup"
    return "explain"


def settings_for(branch):
    """Return the search/prompt settings that fit a given branch."""
    if branch == "compare":
        return {"how_many": 8, "use_memory": False,
                "extra": "Talk about BOTH drinks. Do not answer about only one."}
    if branch == "recommend":
        return {"how_many": 6, "use_memory": True,
                "extra": "Follow the user's stated preferences and say why."}
    if branch == "lookup":
        return {"how_many": 3, "use_memory": False,
                "extra": "Give the exact number and nothing else."}
    return {"how_many": 5, "use_memory": False, "extra": "Explain in three sentences."}


for q in ["What is cold brew", "Compare ashwagandha with turmeric",
          "Suggest one for me", "What is the cold brew ratio"]:
    branch = route(q)
    settings = settings_for(branch)
    print(branch.ljust(10), "pieces:", settings["how_many"],
          " memory:", settings["use_memory"], " <-", q)


## Cell 11 — Using the branch
Same `chat` idea, but the branch decides the settings.

In [ ]:
def chat_with_branch(user_text):
    """Same idea as chat(), but the branch (from route()) decides the search settings."""
    branch = route(user_text)
    settings = settings_for(branch)

    for fact in find_preferences(user_text):
        if fact not in preferences:
            preferences.append(fact)

    hits = search(user_text, settings["how_many"])

    memory_text = "none"
    if settings["use_memory"]:
        memory_text = format_preferences()

    prompt = build_prompt(user_text, hits, memory_text, format_history())
    prompt = prompt + "\n" + settings["extra"]
    reply = ask_model(prompt)

    history.append(["User", user_text])
    history.append(["Assistant", reply])
    print("  (branch:", branch, "| pieces:", len(hits), ")")
    return reply


print("🤖 Answer: ", chat_with_branch("Compare ashwagandha coffee with turmeric coffee"))


In [ ]:
questions = ["Tell me about ashwagandha coffee.",
             "I prefer low caffeine drinks.",
             "Compare it with turmeric coffee.",
             "Compare ashwagandha coffee with turmeric coffee"]

for q in questions:
    print("\n👤 Question:", q)
    print("🤖 Answer: ", chat_with_branch(q))


## Cell 12 — 🪜 LEVEL 5: spend only when needed (Adaptive RAG)
Some questions are easy and some are hard. Running the expensive settings on everything wastes money.

Two clues tell us a question is hard:
1. words like *why*, *compare*, *how* in the question
2. **flat scores** — if the best piece scores about the same as the worst, nothing really matched, so the search is guessing

In [ ]:
def looks_hard(question, hits):
    """Decide whether a question needs the more careful (expensive) settings."""
    q = question.lower()

    if "why" in q or "compare" in q or "how does" in q:
        return True

    if len(hits) < 2:
        return True

    best = hits[0]["score"]
    worst = hits[-1]["score"]
    if best - worst < 0.05:      # flat: nothing stood out
        return True

    return False


def choose_settings(question):
    """Peek at the top results first, then decide: 'quick' or 'careful' settings."""
    first_look = search(question, 8)
    if looks_hard(question, first_look):
        return "careful", 12
    return "quick", 4


for q in ["What is the cold brew ratio",
          "Compare ashwagandha with turmeric coffee",
          "Why is turmeric coffee caffeine free"]:
    style, how_many = choose_settings(q)
    print(style.ljust(9), "pieces:", how_many, " <-", q)


## Cell 13 — 🪜 LEVEL 6: check the answer (Corrective RAG)
Everything so far tried to write a good answer. Nothing has checked whether it **is** good.

Four small checks. Each is a few lines, and each catches a different mistake:

| Check | Question it asks |
|---|---|
| **E** Evidence | did the search find anything decent? |
| **F** Fair citations | do the [ids] in the answer really exist? |
| **C** Coverage | did we answer all parts of the question? |
| **R** Rules | does every sentence have an [id]? |

In [ ]:
EVIDENCE_MIN = 0.30   # minimum similarity score to count a retrieved piece as "good evidence"


def find_citations(answer_text):
    """Pull out every [id] written in the answer. No regex - just splitting."""
    found = []
    pieces = answer_text.split("[")
    for piece in pieces:
        if "]" in piece:
            name = piece.split("]")[0]
            if name not in found:
                found.append(name)
    return found


def check_evidence(hits):
    """E: score how much of the retrieved evidence actually looks relevant."""
    good = 0
    for hit in hits:
        if hit["score"] >= EVIDENCE_MIN:
            good = good + 1
    if good >= 2:
        return 1.0
    if good == 1:
        return 0.5
    return 0.0


def check_citations_real(answer_text, hits):
    """F: check that every [id] cited in the answer actually corresponds to a retrieved piece."""
    real_ids = []
    for hit in hits:
        real_ids.append(hit["id"])

    cited = find_citations(answer_text)
    if len(cited) == 0:
        return 0.0

    good = 0
    for name in cited:
        if name in real_ids:
            good = good + 1
    return good / len(cited)


def check_coverage(answer_text, topics):
    """C: check what fraction of the expected `topics` actually appear in the answer."""
    if len(topics) == 0:
        return 1.0
    found = 0
    for topic in topics:
        if topic.lower() in answer_text.lower():
            found = found + 1
    return found / len(topics)


def check_every_sentence_cited(answer_text):
    """R: check that every non-trivial sentence in the answer carries an [id] citation."""
    for sentence in answer_text.split("."):
        if len(sentence.strip()) < 5:
            continue
        if "[" not in sentence:
            return False
    return True


def check_answer(answer_text, hits, topics):
    """Run all four checks and decide whether the answer passes overall."""
    result = {}
    result["E"] = check_evidence(hits)
    result["F"] = check_citations_real(answer_text, hits)
    result["C"] = check_coverage(answer_text, topics)
    result["R"] = check_every_sentence_cited(answer_text)
    result["passed"] = (result["E"] >= 0.7 and result["F"] >= 0.8
                        and result["C"] >= 0.7 and result["R"] == True)
    return result


## Cell 14 — Four bad answers, four different failures
🎲 **Guess first:** an answer that cites a page that doesn't exist — which letter does it break?

In [ ]:
hits = search("Compare ashwagandha coffee with turmeric coffee", 5)
topics = ["ashwagandha", "turmeric"]

first_id = hits[0]["id"]
second_id = hits[1]["id"]

good_answer  = f"Ashwagandha coffee has caffeine [{first_id}]. Turmeric coffee is caffeine free [{second_id}]."
no_citation  = f"Ashwagandha coffee has caffeine [{first_id}]. Turmeric coffee is caffeine free [{second_id}]. It also cures insomnia."
fake_page    = f"Ashwagandha coffee has caffeine [99_not_a_real_page]. Turmeric coffee is caffeine free [{second_id}]."
only_half    = f"Ashwagandha coffee has caffeine [{first_id}]."

drafts = [["GOOD", good_answer], ["no citation", no_citation],
          ["fake page", fake_page], ["only half", only_half]]

for name, draft in drafts:
    print(name.ljust(14), check_answer(draft, hits, topics))


## Cell 15 — Try again when the check fails
If the answer fails, **search wider and ask again**. Two tries, then stop and say so honestly.

A repair loop with no limit is just an infinite loop.

In [ ]:
def careful_answer(question, topics):
    """If the answer fails the checks, search WIDER and try again - up to 3 tries total."""
    how_many = 4
    tries = 0

    while tries < 3:
        hits = search(question, how_many)
        prompt = build_prompt(question, hits)
        if tries > 0:
            prompt = prompt + "\nPut an [id] on EVERY sentence and cover EVERY part of the question."

        draft = ask_model(prompt)
        result = check_answer(draft, hits, topics)
        print("  try", tries, "| pieces:", how_many, "|", result)

        if result["passed"]:
            return draft

        how_many = how_many * 2   # search wider next time
        tries = tries + 1

    return "I could not find a complete answer in our coffee pages. Which drink should I focus on?"


print("\n🤖", careful_answer("Compare ashwagandha coffee with turmeric coffee",
                              ["ashwagandha", "turmeric"]))


## Cell 16 — What we built

| Level | Idea | What we added |
|---|---|---|
| L1 | Basic RAG | search, then answer from what you found |
| L2 | Template | one prompt in one place, with a "say you don't know" rule |
| L3 | Memory | keep history and preferences in separate slots |
| L4 | Branched | look at the question, then decide how to search |
| L5 | Adaptive | work harder only when the question is hard |
| L6 | Corrective | check the answer, retry if it fails, stop after two tries |

> *"Session 3 found her. Session 4 made it work for millions. Today the app learned to talk — and every step was about one thing: **how do you make a machine that can talk say only true things?** Search first. Write the rules down. Remember the person. Send the question to the right desk. And check your work before you speak."*

**Homework:** add a fifth branch called `safety`. If the question contains "pregnant", "medicine" or "safe", answer from the *Benefits, Cautions & Practical Tips* section only, and add a line saying this is not medical advice.

In [ ]:
SAFETY_KEYWORDS = ["pregnant", "medicine", "safe"]
SAFETY_SECTION = "Benefits, Cautions & Practical Tips"

def is_safety_question(question):
    """Check if the question touches on a safety-sensitive topic."""
    q_lower = question.lower()
    for keyword in SAFETY_KEYWORDS:
        if keyword in q_lower:
            return True
    return False


def search_safety_only(question, how_many=4):
    """Like search(), but only returns hits from the safety section."""
    question_vector = embedder.encode(question).tolist()
    result = index.query(vector=question_vector,
                          top_k=how_many * 3,  # over-fetch since we'll filter down
                          namespace="docs",
                          include_metadata=True)

    hits = []
    for match in result["matches"]:
        if match["metadata"]["section"] == SAFETY_SECTION:
            hit = {}
            hit["id"] = match["id"]
            hit["score"] = round(match["score"], 3)
            hit["title"] = match["metadata"]["title"]
            hit["section"] = match["metadata"]["section"]
            hit["text"] = match["metadata"]["text"]
            hits.append(hit)
        if len(hits) >= how_many:
            break
    return hits


In [ ]:
def route_question(question):
    """L4/L5-style branch, now with a 5th 'safety' branch added."""
    if is_safety_question(question):
        hits = search_safety_only(question)
        prompt = build_prompt(question, hits)
        prompt = prompt + "\nAdd this line at the end: 'This is not medical advice.'"
        return ask_model(prompt)

    # ...existing branches (basic, hard/adaptive, etc.) go here
    hits = search(question)
    prompt = build_prompt(question, hits)
    return ask_model(prompt)


In [ ]:
questions = [
    "Tell me about ashwagandha coffee.",
    "Is ashwagandha coffee safe during pregnancy?",
    "What medicine interactions should I know about?",
    "Compare it with turmeric coffee."
]

for q in questions:
    print("\n👤", q)
    print("🤖", route_question(q))
